In [1]:
# =============================================================================
# nb52 - REGENERATE ALL FIGURES TO SPRINGER ARTWORK SPECIFICATION
#
# Three requirements from the journal's artwork guidance are currently unmet.
#
#   RESOLUTION. Combination artwork, meaning line drawings carrying extensive
#   lettering, requires 600 dpi minimum. The existing figures were written at
#   dpi=140 to 160 and land between 386 and 521 dpi effective. All eight fail.
#
#   LETTERING. Roughly 2-3 mm, that is 8-12 pt, at final printed size, with
#   minimal variation within a figure, in a sans-serif face. Several figures have
#   axis-tick and legend text well below that once scaled to column width.
#
#   NO TITLES INSIDE ARTWORK. The guidance states plainly: do not include titles
#   or captions within your illustrations. The calibration figure currently
#   renders "Calibration of probabilities on the held-out source pool, per class"
#   inside the image, which duplicates the caption and must be removed.
#
# This notebook rebuilds every figure from the committed result CSVs, so nothing
# is re-estimated and no number can change. It only re-renders.
#
# TARGET GEOMETRY. The journal asks for figures 84 mm or 174 mm wide, no taller
# than 234 mm. We render at 174 mm (6.85 in) single-column width and cap height at
# 130 mm, then set dpi=600. Lettering is set in points against that physical size,
# so what you specify is what prints.
# =============================================================================
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, sys, json, shutil, subprocess, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
RD=config.REPORTS_DIR
FIGDIR=PROJECT_ROOT/'figures'/'springer'; FIGDIR.mkdir(parents=True, exist_ok=True)

DPI=900
W_IN=6.85            # 174 mm single-column width
MAXH_IN=5.12         # 130 mm, well inside the 234 mm ceiling
plt.rcParams.update({
 'font.family':'sans-serif',
 'font.sans-serif':['DejaVu Sans','Helvetica','Arial'],
 'font.size':9,          # body lettering, about 3.2 mm
 'axes.titlesize':9.5,
 'axes.labelsize':9,
 'xtick.labelsize':8,    # the floor the guidance allows
 'ytick.labelsize':8,
 'legend.fontsize':8,
 'axes.spines.top':False,
 'axes.spines.right':False,
 'figure.dpi':DPI,
 'savefig.dpi':DPI,
 'savefig.bbox':'tight',
 'savefig.facecolor':'white',
 'lines.linewidth':1.2,
 'axes.linewidth':0.6,   # above the 0.3 pt minimum
 'xtick.major.width':0.6,
 'ytick.major.width':0.6,
})
C={'nslkdd':'#2f4b7c','cicids2017':'#c65911','ugr16':'#4a5a2f','ciciot2023':'#7b3f8f'}
L={'nslkdd':'NSL-KDD','cicids2017':'CIC-IDS2017','ugr16':"UGR'16",'ciciot2023':'CIC-IoT-2023'}

def save(fig, name):
    """Write at DPI and report the effective resolution at the DISPLAY width the
    manuscript uses. bbox_inches='tight' trims the canvas, so the pixel count is
    lower than dpi x figsize; the check below measures what actually ships."""
    path=FIGDIR/name
    fig.savefig(path, dpi=DPI)
    plt.close(fig)
    import struct
    d=open(path,'rb').read(33); px,py=struct.unpack('>II', d[16:24])
    eff=px/W_IN                      # every figure is displayed at W_IN
    h_mm=py/eff*25.4
    flag='PASS' if eff>=600 and h_mm<=234 else 'FAIL'
    print(f"  {name:12s} {px}x{py:<6} {eff:5.0f} dpi at {W_IN*25.4:.0f} mm  h={h_mm:5.1f} mm  [{flag}]")
    return eff

print(f'ready | target {DPI} dpi at {W_IN:.2f} in wide ({W_IN*25.4:.0f} mm)')


Mounted at /content/drive
ready | target 900 dpi at 6.85 in wide (174 mm)


In [2]:
# =============================================================================
# Cell 2 - Fig 4 (alpha sensitivity) and Fig 5 (efficiency).
# Column names taken from the committed CSVs, not assumed.
# =============================================================================
t0=time.time()

# alpha_sensitivity.csv columns: dataset, focal_class, alpha, nominal, REC, TSC, SHC, ...
a=pd.read_csv(RD/'alpha_sensitivity.csv')
fig,ax=plt.subplots(figsize=(W_IN, min(W_IN*0.52, MAXH_IN)))
alphas=sorted(a['alpha'].unique())
ax.plot(alphas, [1-x for x in alphas], ls='--', color='#666', lw=1.0, label=r'nominal $1-\alpha$')
for (ds,cl),g in a.groupby(['dataset','focal_class']):
    g=g.sort_values('alpha')
    ax.plot(g['alpha'], g['SHC'], marker='o', ms=3.5,
            color=C.get(ds,'#333'), label=f"{L.get(ds,ds)} {cl}")
ax.set_xlabel(r'miscoverage level $\alpha$')
ax.set_ylabel('focal SHC coverage')
ax.set_xticks(alphas); ax.set_ylim(-0.03, 1.05)
ax.legend(loc='center left', bbox_to_anchor=(1.01,0.5), frameon=False, handlelength=1.6)
save(fig, 'Fig4.png')

# efficiency_setsize.csv columns: dataset, protocol, focal, focal_coverage, focal_set_size, ...
e=pd.read_csv(RD/'efficiency_setsize.csv')
fig,ax=plt.subplots(figsize=(W_IN, min(W_IN*0.52, MAXH_IN)))
MK={'nslkdd':'o','cicids2017':'s','ugr16':'^','ciciot2023':'D'}
PC={'SHC':'#b3261e','TSC':'#2e7d32','REC':'#616161'}
ax.axhline(0.95, ls='--', color='#999', lw=0.9)
for (ds,pr),g in e.groupby(['dataset','protocol']):
    ax.scatter(g['focal_set_size'], g['focal_coverage'], marker=MK.get(ds,'o'), s=34,
               color=PC.get(pr,'#333'), edgecolor='white', lw=0.4, zorder=3)
h1=[plt.Line2D([],[],marker=MK[d],ls='',color='#555',ms=5,label=L[d]) for d in MK if d in e.dataset.values]
h2=[plt.Line2D([],[],marker='o',ls='',color=PC[q],ms=5,label=q) for q in PC if q in e.protocol.values]
lg=ax.legend(handles=h1, loc='upper left', bbox_to_anchor=(1.01,1.0), frameon=False, title='dataset')
plt.setp(lg.get_title(), fontsize=8); ax.add_artist(lg)
lg2=ax.legend(handles=h2, loc='upper left', bbox_to_anchor=(1.01,0.45), frameon=False, title='protocol')
plt.setp(lg2.get_title(), fontsize=8)
ax.set_xlabel('focal prediction-set size'); ax.set_ylabel('focal coverage')
ax.set_ylim(-0.05, 1.08)
save(fig, 'Fig5.png')
print(f'  [{time.time()-t0:.0f}s]')


  Fig4.png     7364x2994    1075 dpi at 174 mm  h= 70.7 mm  [PASS]
  Fig5.png     6028x2990     880 dpi at 174 mm  h= 86.3 mm  [PASS]
  [5s]


In [ ]:
# =============================================================================
# Cell 3 - Fig 6 (calibration). The internal title is REMOVED: the artwork
# guidance prohibits titles inside illustrations and the caption carries it.
# calibration_quality.csv columns: dataset, class, brier, ece
# =============================================================================
c=pd.read_csv(RD/'calibration_quality.csv')
fig,axs=plt.subplots(1,2, figsize=(W_IN, min(W_IN*0.40, MAXH_IN)))
order=[d for d in ['nslkdd','cicids2017','ugr16','ciciot2023'] if d in c.dataset.values]
for ax,(col,lab,thr) in zip(axs, [('ece','expected calibration error',0.002),
                                  ('brier','Brier score',None)]):
    x=0; ticks=[]; labels=[]
    for ds in order:
        g=c[c.dataset==ds]
        for _,r in g.iterrows():
            ax.bar(x, r[col], color=C[ds], width=0.8, edgecolor='none')
            ticks.append(x); labels.append(str(r['class'])[:9]); x+=1
        x+=0.7
    if thr is not None:
        ax.axhline(thr, ls='--', color='#b3261e', lw=0.9)
        ax.text(0.01, thr, f'{thr}', transform=ax.get_yaxis_transform(),
                fontsize=7, color='#b3261e', va='bottom')
    ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=90, fontsize=6.5)
    ax.set_ylabel(lab)
    # no set_title and no suptitle: titles inside artwork are prohibited
h=[plt.Line2D([],[],marker='s',ls='',color=C[d],ms=5,label=L[d]) for d in order]
axs[1].legend(handles=h, loc='upper right', frameon=False)
fig.tight_layout()
save(fig, 'Fig6.png')
print("  internal title removed; the (a)/(b) panel labels belong in the caption")


In [ ]:
# =============================================================================
# Cell 4 - Fig 3 mechanism, Fig 7 monitor, Fig 2 placebo, Fig 8 selective.
# =============================================================================
t0=time.time()

# threshold_displacement_class_level.csv carries the class-level mechanism points
m=pd.read_csv(RD/'threshold_displacement_class_level.csv')
fig,ax=plt.subplots(figsize=(W_IN, min(W_IN*0.46, MAXH_IN)))
for ds,g in m.groupby('dataset'):
    ax.scatter(g['KS_two_sided'], g['undercoverage'], color=C.get(ds,'#333'), s=34,
               edgecolor='white', lw=0.4, label=L.get(ds,ds), zorder=3)
ax.axhline(0, ls=':', color='#999', lw=0.9)
ax.set_xlabel('true-class score movement (KS distance)')
ax.set_ylabel('undercoverage')
ax.legend(frameon=False, loc='upper left')
save(fig, 'Fig3.png')

# monitor
mo=pd.read_csv(RD/'monitor_labelfree.csv')
mo['ds']=mo['dataset'].str.split(':').str[0]
agg=mo.groupby(['ds','class'],as_index=False).agg(
    risk=('drift_labelfree','mean'), under=('undercoverage','mean'))
fig,ax=plt.subplots(figsize=(W_IN, min(W_IN*0.42, MAXH_IN)))
for ds,g in agg.groupby('ds'):
    ax.scatter(g['risk'], g['under'], color=C.get(ds,'#333'), s=34,
               edgecolor='white', lw=0.4, label=L.get(ds,ds), zorder=3)
ax.axhline(0.05, ls=':', color='#999', lw=0.9)
ax.set_xlabel('label-free monitor signal'); ax.set_ylabel('undercoverage')
ax.legend(frameon=False, loc='upper left')
save(fig, 'Fig7.png')

# placebo ladder: placebo_ladder_curve.csv -> rung, coverage, S_sup, score_KS
pl=pd.read_csv(RD/'placebo_ladder_curve.csv').sort_values('rung')
fig,axs=plt.subplots(1,2, figsize=(W_IN, min(W_IN*0.36, MAXH_IN)))
axs[0].plot(pl['rung'], pl['coverage'], marker='o', ms=4, color='#b3261e',
            label='placebo (seen subtypes)')
axs[0].set_xlabel('share of focal mass from guess_passwd')
axs[0].set_ylabel('focal coverage'); axs[0].legend(frameon=False)
axs[1].plot(pl['rung'], pl['score_KS'], marker='o', ms=4, color='#4a5a2f')
axs[1].set_xlabel('share of focal mass from guess_passwd')
axs[1].set_ylabel('score movement (KS)')
fig.tight_layout()
save(fig, 'Fig2.png')

# selective prediction on the failing UGR classes
sp=pd.read_csv(RD/'selective_prediction_ugr_curve.csv')
fig,ax=plt.subplots(figsize=(W_IN, min(W_IN*0.42, MAXH_IN)))
for cl,g in sp.groupby('class'):
    g=g.sort_values('escalated_frac')
    ax.plot(g['escalated_frac'], g['coverage_retained'], marker='o', ms=3.5, label=str(cl))
ax.axhline(0.95, ls='--', color='#999', lw=0.9)
ax.set_xlabel('fraction of alerts escalated'); ax.set_ylabel('retained-set coverage')
ax.legend(frameon=False, loc='lower right', ncol=2)
save(fig, 'Fig8.png')
print(f'  [{time.time()-t0:.0f}s]')


In [ ]:
# =============================================================================
# Cell 5 - verify every figure against the artwork specification, then commit.
# =============================================================================
import struct
print("SPRINGER ARTWORK COMPLIANCE")
print(f"{'file':14s} {'px':>13s} {'dpi at 174mm':>13s} {'height mm':>10s} {'verdict':>9s}")
rows=[]
for f in sorted(os.listdir(FIGDIR)):
    if not f.endswith('.png'): continue
    d=open(FIGDIR/f,'rb').read(33); px,py=struct.unpack('>II', d[16:24])
    eff=px/W_IN; h_mm=py/eff*25.4
    ok = eff>=600 and h_mm<=234
    rows.append({'file':f,'px_w':px,'px_h':py,'dpi':round(eff),'height_mm':round(h_mm,1),'compliant':ok})
    print(f"  {f:12s} {px}x{py:<7} {eff:13.0f} {h_mm:10.1f} {'PASS' if ok else 'FAIL':>9s}")
V=pd.DataFrame(rows)
print(f"\n  compliant: {int(V.compliant.sum())}/{len(V)}")
print(f"  minimum dpi: {V.dpi.min()} (requirement 600)")
print(f"  tallest: {V.height_mm.max()} mm (ceiling 234)")
V.to_csv(RD/'figure_compliance_springer.csv', index=False)

print("\n  Figures are named Fig2.png through Fig8.png per the naming convention.")
print("  Fig1 is the schematic overview and is not regenerated here; redraw it in a")
print("  vector tool and export EPS, which the guidance prefers for vector graphics.")

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','nb52: regenerate figures at 600 dpi with Springer-compliant lettering; remove title from inside Fig 6')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
